# Lab 02 - Mục tiêu 2: Giáo dục

**Thành viên phụ trách:** Hoàng Quốc Việt (23120189)

Notebook này thực hiện các công việc của thành viên Quốc Việt, trừ phần báo cáo chính:

- **2.1:** Khảo sát các indicator giáo dục trong WDI và xác nhận mã chỉ số khả dụng.
- **2.2:** Tiền xử lý dữ liệu giáo dục: lọc indicator, loại aggregate rows, xử lý missing values theo quy ước nhóm, chuẩn hóa cột và xuất file `wdi_education.csv`.
- **2.3:** Giải thích ý nghĩa từng metric và lý do chọn.
- **2.4 - 2.5:** Chuẩn bị kế hoạch worksheet Tableau và hướng nhận xét cần kiểm tra.
- **2.7:** Tính thống kê mô tả cơ bản cho các indicator giáo dục.


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)


In [11]:
cwd = Path.cwd()
if (cwd / "WDIEXCEL.xlsx").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "WDIEXCEL.xlsx").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Không tìm thấy WDIEXCEL.xlsx ở thư mục hiện tại hoặc thư mục cha.")

SOURCE_FILE = PROJECT_ROOT / "WDIEXCEL.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data_output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Thư mục gốc dự án: {PROJECT_ROOT}")
print(f"File dữ liệu nguồn: {SOURCE_FILE.name}")
print(f"Thư mục đầu ra: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


Thư mục gốc dự án: d:\Lab2_TQH
File dữ liệu nguồn: WDIEXCEL.xlsx
Thư mục đầu ra: data_output


## 2.1. Khảo sát indicator giáo dục

Bốn indicator được chọn đều thuộc WDI và phù hợp với mục tiêu phân tích về tiếp cận giáo dục, kết quả giáo dục và đầu tư cho giáo dục:

| Indicator Code | Cột sau tiền xử lý | Ý nghĩa trong phân tích |
|---|---|---|
| `SE.PRM.NENR` | `Primary_Net_Enrollment` | Tỷ lệ nhập học đúng độ tuổi ở bậc tiểu học |
| `SE.SEC.NENR` | `Secondary_Net_Enrollment` | Tỷ lệ nhập học đúng độ tuổi ở bậc trung học |
| `SE.ADT.LITR.ZS` | `Adult_Literacy_Rate` | Tỷ lệ biết chữ của người từ 15 tuổi trở lên |
| `SE.XPD.TOTL.GD.ZS` | `Education_Expenditure_GDP` | Chi tiêu giáo dục của chính phủ theo % GDP |


In [12]:
INDICATORS = {
    "SE.PRM.NENR": {
        "clean_name": "Primary_Net_Enrollment",
        "analysis_role": "Mức độ tiếp cận giáo dục tiểu học",
    },
    "SE.SEC.NENR": {
        "clean_name": "Secondary_Net_Enrollment",
        "analysis_role": "Mức độ tiếp cận giáo dục trung học",
    },
    "SE.ADT.LITR.ZS": {
        "clean_name": "Adult_Literacy_Rate",
        "analysis_role": "Kết quả giáo dục qua tỷ lệ biết chữ",
    },
    "SE.XPD.TOTL.GD.ZS": {
        "clean_name": "Education_Expenditure_GDP",
        "analysis_role": "Mức đầu tư của chính phủ cho giáo dục",
    },
}

indicator_codes = list(INDICATORS)
years = [str(year) for year in range(2000, 2024)]

series_cols = [
    "Series Code",
    "Topic",
    "Indicator Name",
    "Long definition",
    "Unit of measure",
    "Periodicity",
    "Source",
]
series = pd.read_excel(SOURCE_FILE, sheet_name="Series", usecols=series_cols)
indicator_info = (
    series[series["Series Code"].isin(indicator_codes)]
    .copy()
    .assign(
        Clean_Column=lambda df: df["Series Code"].map(
            {code: meta["clean_name"] for code, meta in INDICATORS.items()}
        ),
        Analysis_Role=lambda df: df["Series Code"].map(
            {code: meta["analysis_role"] for code, meta in INDICATORS.items()}
        ),
        Available=True,
    )
    .sort_values("Series Code")
)

missing_codes = sorted(set(indicator_codes) - set(indicator_info["Series Code"]))
if missing_codes:
    raise ValueError(f"Các indicator không có trong sheet Series: {missing_codes}")

indicator_info


,Series Code,Topic,Indicator Name,Long definition,Unit of measure,Periodicity,Source,Clean_Column,Analysis_Role,Available
802,SE.ADT.LITR.ZS,Education: Outcomes,"Literacy rate, adult total (% of people ages 15 and above)",Adult literacy rate is the percentage of people ages 15 and above who can both read and write with understanding a short simple statemen...,% of people ages 15 and above,Annual,"Data API, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://databrowser.uis.unesco.org/resources, note: The da...",Adult_Literacy_Rate,Kết quả giáo dục qua tỷ lệ biết chữ,True
842,SE.PRM.NENR,Education: Participation,"School enrollment, primary (% net)",Net enrollment rate is the ratio of children of official school age who are enrolled in school to the population of the corresponding of...,NaN,Annual,"Stat Bulk Data Download Service, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://uis.unesco.org/bdds, publis...",Primary_Net_Enrollment,Mức độ tiếp cận giáo dục tiểu học,True
901,SE.SEC.NENR,Education: Participation,"School enrollment, secondary (% net)",Net enrollment rate is the ratio of children of official school age who are enrolled in school to the population of the corresponding of...,NaN,Annual,"Stat Bulk Data Download Service, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://uis.unesco.org/bdds, publis...",Secondary_Net_Enrollment,Mức độ tiếp cận giáo dục trung học,True
951,SE.XPD.TOTL.GD.ZS,Education: Inputs,"Government expenditure on education, total (% of GDP)","General government expenditure on education (current, capital, and transfers) is expressed as a percentage of GDP. It includes expenditu...",% of GDP,Annual,"Data API, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://databrowser.uis.unesco.org/resources, note: The da...",Education_Expenditure_GDP,Mức đầu tư của chính phủ cho giáo dục,True


In [13]:
country_cols = ["Country Code", "Short Name", "Table Name", "Region", "Income Group"]
country = pd.read_excel(SOURCE_FILE, sheet_name="Country", usecols=country_cols)

country_clean = country.rename(
    columns={
        "Country Code": "Country_Code",
        "Short Name": "Short_Name",
        "Table Name": "Country_Name_Table",
        "Income Group": "Income_Group",
    }
)

# Trong WDI, các dòng aggregate như World, income groups, regions thường có Region bị trống.
real_countries = country_clean[country_clean["Region"].notna()].copy()
aggregate_rows = country_clean[country_clean["Region"].isna()].copy()

print(f"Số dòng metadata quốc gia: {len(country_clean):,}")
print(f"Số quốc gia thật được giữ lại: {len(real_countries):,}")
print(f"Số dòng aggregate bị loại: {len(aggregate_rows):,}")

region_summary = real_countries["Region"].value_counts().rename_axis("Region").reset_index(name="Country_Count")
region_summary


Số dòng metadata quốc gia: 265
Số quốc gia thật được giữ lại: 217
Số dòng aggregate bị loại: 48


,Region,Country_Count
0,Europe & Central Asia,58
1,Sub-Saharan Africa,48
2,Latin America & Caribbean,42
3,East Asia & Pacific,37
4,Middle East & North Africa,23
5,South Asia,6
6,North America,3


In [14]:
data_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"] + years
raw = pd.read_excel(SOURCE_FILE, sheet_name="Data", usecols=data_cols)

education_raw = raw[
    raw["Indicator Code"].isin(indicator_codes)
    & raw["Country Code"].isin(real_countries["Country_Code"])
].copy()

availability_rows = []
for indicator_code in indicator_codes:
    sub = education_raw[education_raw["Indicator Code"].eq(indicator_code)].copy()
    values = sub[years]
    yearly_non_null = values.notna().sum()
    availability_rows.append(
        {
            "Indicator_Code": indicator_code,
            "Clean_Column": INDICATORS[indicator_code]["clean_name"],
            "Indicator_Name": sub["Indicator Name"].iloc[0] if len(sub) else None,
            "Country_Rows": len(sub),
            "Countries_With_Any_Data": int(values.notna().any(axis=1).sum()),
            "Non_Null_Cells_2000_2023": int(values.notna().sum().sum()),
            "First_Year_With_Data": int(yearly_non_null[yearly_non_null.gt(0)].index.min()),
            "Last_Year_With_Data": int(yearly_non_null[yearly_non_null.gt(0)].index.max()),
        }
    )

availability = pd.DataFrame(availability_rows)
availability


,Indicator_Code,Clean_Column,Indicator_Name,Country_Rows,Countries_With_Any_Data,Non_Null_Cells_2000_2023,First_Year_With_Data,Last_Year_With_Data
0,SE.PRM.NENR,Primary_Net_Enrollment,"School enrollment, primary (% net)",217,188,2419,2000,2019
1,SE.SEC.NENR,Secondary_Net_Enrollment,"School enrollment, secondary (% net)",217,180,1891,2000,2019
2,SE.ADT.LITR.ZS,Adult_Literacy_Rate,"Literacy rate, adult total (% of people ages 15 and above)",217,155,892,2000,2023
3,SE.XPD.TOTL.GD.ZS,Education_Expenditure_GDP,"Government expenditure on education, total (% of GDP)",217,200,3325,2000,2023


## 2.2. Quy ước tiền xử lý

Các bước tiền xử lý:

1. Chỉ giữ 4 indicator giáo dục đã xác minh ở mục 2.1.
2. Chỉ giữ các quốc gia thật, loại aggregate rows bằng metadata `Country.Region`.
3. Giới hạn giai đoạn phân tích `2000-2023`.
4. Với từng cặp `Country_Code` và `Indicator_Code`, chỉ nội suy tuyến tính các khoảng missing **bên trong chuỗi** có độ dài tối đa 2 năm.
5. Nếu một cặp country-indicator có khoảng missing bên trong dài hơn 2 năm thì loại cặp đó khỏi dữ liệu sạch để tránh tạo dữ liệu giả.
6. Không extrapolate cho các năm đầu/cuối nằm ngoài phạm vi quan sát của indicator.
7. Xuất dữ liệu dạng wide để nạp vào Tableau, mỗi dòng là một `Country_Code` - `Year`.


In [15]:
def max_missing_run(values: pd.Series) -> int:
    max_run = 0
    current = 0
    for value in values:
        if pd.isna(value):
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run


def max_internal_missing_run(values: pd.Series) -> int:
    observed_positions = np.flatnonzero(values.notna().to_numpy())
    if len(observed_positions) == 0:
        return -1
    if len(observed_positions) == 1:
        return 0
    internal = values.iloc[observed_positions[0] : observed_positions[-1] + 1]
    return max_missing_run(internal)


def clean_country_indicator(row: pd.Series) -> tuple[str, pd.Series, int]:
    values = pd.to_numeric(row[years], errors="coerce")
    observed_positions = np.flatnonzero(values.notna().to_numpy())

    if len(observed_positions) == 0:
        return "dropped_no_data", values, 0

    max_gap = max_internal_missing_run(values)
    if max_gap > 2:
        return "dropped_internal_gap_gt_2", values, 0

    cleaned = values.copy()
    first_pos = observed_positions[0]
    last_pos = observed_positions[-1]
    internal = cleaned.iloc[first_pos : last_pos + 1]
    interpolated_internal = internal.interpolate(method="linear", limit=2, limit_area="inside")
    cleaned.iloc[first_pos : last_pos + 1] = interpolated_internal

    imputed_count = int(values.isna().sum() - cleaned.isna().sum())
    return "kept", cleaned, imputed_count


In [16]:
long_records = []
cleaning_records = []

for _, row in education_raw.iterrows():
    status, cleaned_values, imputed_count = clean_country_indicator(row)
    original_values = pd.to_numeric(row[years], errors="coerce")
    max_internal_gap = max_internal_missing_run(original_values)

    cleaning_records.append(
        {
            "Country_Code": row["Country Code"],
            "Country_Name": row["Country Name"],
            "Indicator_Code": row["Indicator Code"],
            "Indicator_Name": row["Indicator Name"],
            "Clean_Column": INDICATORS[row["Indicator Code"]]["clean_name"],
            "Status": status,
            "Observed_Cells_Before": int(original_values.notna().sum()),
            "Observed_Cells_After": int(cleaned_values.notna().sum()) if status == "kept" else 0,
            "Imputed_Cells": imputed_count if status == "kept" else 0,
            "Max_Internal_Missing_Run": max_internal_gap,
        }
    )

    if status != "kept":
        continue

    value_col = INDICATORS[row["Indicator Code"]]["clean_name"]
    for year, value in cleaned_values.items():
        if pd.notna(value):
            long_records.append(
                {
                    "Country_Name": row["Country Name"],
                    "Country_Code": row["Country Code"],
                    "Year": int(year),
                    "Indicator_Code": row["Indicator Code"],
                    "Indicator_Name": row["Indicator Name"],
                    "Metric": value_col,
                    "Value": float(value),
                }
            )

education_long = pd.DataFrame(long_records)
cleaning_report = pd.DataFrame(cleaning_records)

status_summary = (
    cleaning_report
    .groupby(["Indicator_Code", "Clean_Column", "Status"], as_index=False)
    .agg(
        Country_Indicator_Count=("Country_Code", "count"),
        Observed_Cells_Before=("Observed_Cells_Before", "sum"),
        Observed_Cells_After=("Observed_Cells_After", "sum"),
        Imputed_Cells=("Imputed_Cells", "sum"),
    )
)
status_summary


,Indicator_Code,Clean_Column,Status,Country_Indicator_Count,Observed_Cells_Before,Observed_Cells_After,Imputed_Cells
0,SE.ADT.LITR.ZS,Adult_Literacy_Rate,dropped_internal_gap_gt_2,114,646,0,0
1,SE.ADT.LITR.ZS,Adult_Literacy_Rate,dropped_no_data,62,0,0,0
2,SE.ADT.LITR.ZS,Adult_Literacy_Rate,kept,41,246,294,48
3,SE.PRM.NENR,Primary_Net_Enrollment,dropped_internal_gap_gt_2,45,412,0,0
4,SE.PRM.NENR,Primary_Net_Enrollment,dropped_no_data,29,0,0,0
5,SE.PRM.NENR,Primary_Net_Enrollment,kept,143,2007,2144,137
6,SE.SEC.NENR,Secondary_Net_Enrollment,dropped_internal_gap_gt_2,47,372,0,0
7,SE.SEC.NENR,Secondary_Net_Enrollment,dropped_no_data,37,0,0,0
8,SE.SEC.NENR,Secondary_Net_Enrollment,kept,133,1519,1637,118
9,SE.XPD.TOTL.GD.ZS,Education_Expenditure_GDP,dropped_internal_gap_gt_2,61,847,0,0


In [17]:
education_wide = (
    education_long
    .pivot_table(
        index=["Country_Name", "Country_Code", "Year"],
        columns="Metric",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)
education_wide.columns.name = None

education_wide = education_wide.merge(
    real_countries[["Country_Code", "Region", "Income_Group"]],
    on="Country_Code",
    how="left",
)

metric_columns = [INDICATORS[code]["clean_name"] for code in indicator_codes]
for col in metric_columns:
    if col not in education_wide.columns:
        education_wide[col] = np.nan

education_wide = education_wide[
    ["Country_Name", "Country_Code", "Region", "Income_Group", "Year"] + metric_columns
].sort_values(["Region", "Country_Name", "Year"]).reset_index(drop=True)

education_wide.to_csv(OUTPUT_DIR / "wdi_education.csv", index=False, encoding="utf-8-sig")

print(f"Số dòng đã xuất: {len(education_wide):,}")
print(f"Số quốc gia đã xuất: {education_wide['Country_Code'].nunique():,}")
print(f"Giai đoạn dữ liệu: {education_wide['Year'].min()}-{education_wide['Year'].max()}")
print(f"File CSV chính: {OUTPUT_DIR / 'wdi_education.csv'}")

education_wide.head(10)


Số dòng đã xuất: 3,537
Số quốc gia đã xuất: 192
Giai đoạn dữ liệu: 2000-2023
File CSV chính: d:\Lab2_TQH\data_output\wdi_education.csv


,Country_Name,Country_Code,Region,Income_Group,Year,Primary_Net_Enrollment,Secondary_Net_Enrollment,Adult_Literacy_Rate,Education_Expenditure_GDP
0,American Samoa,ASM,East Asia & Pacific,High income,2006,NaN,NaN,NaN,14.71705
1,Australia,AUS,East Asia & Pacific,High income,2000,94.39800,NaN,NaN,NaN
2,Australia,AUS,East Asia & Pacific,High income,2001,93.85651,NaN,NaN,NaN
3,Australia,AUS,East Asia & Pacific,High income,2002,93.85948,NaN,NaN,NaN
4,Australia,AUS,East Asia & Pacific,High income,2003,94.85270,NaN,NaN,NaN
5,Australia,AUS,East Asia & Pacific,High income,2004,95.22550,NaN,NaN,NaN
6,Australia,AUS,East Asia & Pacific,High income,2005,95.50656,NaN,NaN,NaN
7,Australia,AUS,East Asia & Pacific,High income,2006,95.63832,NaN,NaN,NaN
8,Australia,AUS,East Asia & Pacific,High income,2007,96.41764,NaN,NaN,NaN
9,Australia,AUS,East Asia & Pacific,High income,2008,96.53968,NaN,NaN,NaN


In [18]:
final_quality = pd.DataFrame(
    [
        {
            "Column": col,
            "Non_Null_Cells": int(education_wide[col].notna().sum()),
            "Countries_With_Data": int(education_wide.loc[education_wide[col].notna(), "Country_Code"].nunique()),
            "Min_Value": float(education_wide[col].min(skipna=True)),
            "Max_Value": float(education_wide[col].max(skipna=True)),
        }
        for col in metric_columns
    ]
)

final_quality


,Column,Non_Null_Cells,Countries_With_Data,Min_Value,Max_Value
0,Primary_Net_Enrollment,2144,143,26.827810,99.944810
1,Secondary_Net_Enrollment,1637,133,3.279680,99.911640
2,Adult_Literacy_Rate,294,41,26.830000,100.000000
3,Education_Expenditure_GDP,2610,139,0.000004,15.515512


## 2.3. Giải thích ý nghĩa metric và lý do chọn

Phần này phục vụ công việc 2.3 của thành viên Quốc Việt. Các chỉ số được chọn đều thuộc nhóm Education trong WDI, có ý nghĩa trực tiếp với câu hỏi phân tích về mức độ tiếp cận giáo dục, kết quả giáo dục và mức đầu tư cho giáo dục.


In [19]:
metric_reason = pd.DataFrame([
    {
        "Indicator_Code": "SE.PRM.NENR",
        "Clean_Column": "Primary_Net_Enrollment",
        "Ý_nghĩa": "Tỷ lệ trẻ em đúng độ tuổi tiểu học đang đi học tiểu học.",
        "Lý_do_chọn": "Đo mức độ tiếp cận giáo dục nền tảng; phù hợp để phân tích xu hướng phổ cập giáo dục cơ bản.",
        "Biểu_đồ_phù_hợp": "Line chart, heatmap",
    },
    {
        "Indicator_Code": "SE.SEC.NENR",
        "Clean_Column": "Secondary_Net_Enrollment",
        "Ý_nghĩa": "Tỷ lệ học sinh đúng độ tuổi trung học đang đi học trung học.",
        "Lý_do_chọn": "Cho thấy khả năng duy trì học tập sau bậc tiểu học; thường phân hóa rõ hơn giữa quốc gia và vùng.",
        "Biểu_đồ_phù_hợp": "Line chart, scatter plot, heatmap",
    },
    {
        "Indicator_Code": "SE.ADT.LITR.ZS",
        "Clean_Column": "Adult_Literacy_Rate",
        "Ý_nghĩa": "Tỷ lệ người từ 15 tuổi trở lên có khả năng đọc và viết một câu đơn giản.",
        "Lý_do_chọn": "Đại diện cho kết quả giáo dục dài hạn; dùng để đối chiếu với enrollment và chi tiêu giáo dục.",
        "Biểu_đồ_phù_hợp": "Scatter plot, heatmap nếu dữ liệu đủ dày",
    },
    {
        "Indicator_Code": "SE.XPD.TOTL.GD.ZS",
        "Clean_Column": "Education_Expenditure_GDP",
        "Ý_nghĩa": "Chi tiêu của chính phủ cho giáo dục, tính theo phần trăm GDP.",
        "Lý_do_chọn": "Phản ánh mức ưu tiên đầu tư của chính phủ cho giáo dục; dùng để so sánh vùng và nhóm thu nhập.",
        "Biểu_đồ_phù_hợp": "Bar chart, scatter plot",
    },
])
metric_reason


,Indicator_Code,Clean_Column,Ý_nghĩa,Lý_do_chọn,Biểu_đồ_phù_hợp
0,SE.PRM.NENR,Primary_Net_Enrollment,Tỷ lệ trẻ em đúng độ tuổi tiểu học đang đi học tiểu học.,Đo mức độ tiếp cận giáo dục nền tảng; phù hợp để phân tích xu hướng phổ cập giáo dục cơ bản.,"Line chart, heatmap"
1,SE.SEC.NENR,Secondary_Net_Enrollment,Tỷ lệ học sinh đúng độ tuổi trung học đang đi học trung học.,Cho thấy khả năng duy trì học tập sau bậc tiểu học; thường phân hóa rõ hơn giữa quốc gia và vùng.,"Line chart, scatter plot, heatmap"
2,SE.ADT.LITR.ZS,Adult_Literacy_Rate,Tỷ lệ người từ 15 tuổi trở lên có khả năng đọc và viết một câu đơn giản.,Đại diện cho kết quả giáo dục dài hạn; dùng để đối chiếu với enrollment và chi tiêu giáo dục.,"Scatter plot, heatmap nếu dữ liệu đủ dày"
3,SE.XPD.TOTL.GD.ZS,Education_Expenditure_GDP,"Chi tiêu của chính phủ cho giáo dục, tính theo phần trăm GDP.",Phản ánh mức ưu tiên đầu tư của chính phủ cho giáo dục; dùng để so sánh vùng và nhóm thu nhập.,"Bar chart, scatter plot"


## 2.7. Thống kê mô tả cơ bản

Phần này phục vụ công việc chung 2.7: tính các thống kê mô tả như số lượng giá trị hợp lệ, trung bình, trung vị, độ lệch chuẩn, min và max cho các indicator giáo dục sau tiền xử lý.


In [20]:
descriptive_stats = (
    education_wide[metric_columns]
    .describe()
    .transpose()
    .rename(columns={
        "count": "Số_giá_trị_hợp_lệ",
        "mean": "Trung_bình",
        "std": "Độ_lệch_chuẩn",
        "min": "Nhỏ_nhất",
        "25%": "Q1",
        "50%": "Trung_vị",
        "75%": "Q3",
        "max": "Lớn_nhất",
    })
)

descriptive_stats


,Số_giá_trị_hợp_lệ,Trung_bình,Độ_lệch_chuẩn,Nhỏ_nhất,Q1,Trung_vị,Q3,Lớn_nhất
Primary_Net_Enrollment,2144.0,89.138014,12.229455,26.827810,87.325757,93.364805,96.669722,99.944810
Secondary_Net_Enrollment,1637.0,72.514201,23.343638,3.279680,62.005060,80.565290,90.064580,99.911640
Adult_Literacy_Rate,294.0,89.832738,11.433459,26.830000,88.461697,93.826669,96.047503,100.000000
Education_Expenditure_GDP,2610.0,4.390010,1.831352,0.000004,3.107373,4.253820,5.404381,15.515512


In [21]:
region_year_stats = (
    education_wide
    .groupby(["Region", "Year"], as_index=False)[metric_columns]
    .mean(numeric_only=True)
    .sort_values(["Region", "Year"])
)

# Bảng này giúp dựng bar chart theo vùng với filter năm trong Tableau.
region_year_stats.tail(14)


,Region,Year,Primary_Net_Enrollment,Secondary_Net_Enrollment,Adult_Literacy_Rate,Education_Expenditure_GDP
153,Sub-Saharan Africa,2010,77.050788,32.280795,78.513000,3.917185
154,Sub-Saharan Africa,2011,77.498561,35.941515,88.696493,4.169977
155,Sub-Saharan Africa,2012,78.288703,37.673551,89.530003,3.942438
156,Sub-Saharan Africa,2013,78.702430,38.677298,89.875835,3.776510
157,Sub-Saharan Africa,2014,78.608095,38.424354,90.221667,3.760557
158,Sub-Saharan Africa,2015,78.335390,38.246284,90.580002,4.010784
159,Sub-Saharan Africa,2016,78.678226,38.835193,91.739998,3.940604
160,Sub-Saharan Africa,2017,78.645644,38.372112,82.830004,3.735163
161,Sub-Saharan Africa,2018,80.964505,41.487965,85.323332,3.927199
162,Sub-Saharan Africa,2019,86.158230,NaN,81.335735,3.975084


## 2.4 và 2.5. Kế hoạch worksheet Tableau và nhận xét cần kiểm tra

Không tạo biểu đồ cuối cùng bằng Python vì đề bài yêu cầu toàn bộ trực quan hóa cuối cùng phải thực hiện bằng Tableau. Bảng dưới đây là đặc tả ngắn để dựng worksheet trong Tableau; bản chi tiết nằm ở `tableau_specs/QuocViet_GiaoDuc_Worksheet_Spec.md`.


In [22]:
worksheet_plan = pd.DataFrame([
    {
        "Worksheet": "EDU_01_Enrollment_Trend",
        "Loại_biểu_đồ": "Line chart",
        "Trường_chính": "Year, Primary_Net_Enrollment, Secondary_Net_Enrollment",
        "Mục_đích": "Theo dõi xu hướng nhập học tiểu học và trung học theo thời gian.",
        "Nhận_xét_cần_kiểm_tra": "Tiểu học thường cao hơn trung học; khoảng cách giữa hai đường cho thấy rào cản chuyển tiếp.",
    },
    {
        "Worksheet": "EDU_02_Expenditure_By_Region",
        "Loại_biểu_đồ": "Bar chart",
        "Trường_chính": "Region, Education_Expenditure_GDP, Year",
        "Mục_đích": "So sánh mức chi tiêu giáo dục giữa các vùng hoặc nhóm thu nhập.",
        "Nhận_xét_cần_kiểm_tra": "Vùng chi tiêu cao chưa chắc có literacy/enrollment cao nếu dữ liệu có độ trễ hoặc thiếu.",
    },
    {
        "Worksheet": "EDU_03_Literacy_vs_Secondary",
        "Loại_biểu_đồ": "Scatter plot",
        "Trường_chính": "Secondary_Net_Enrollment, Adult_Literacy_Rate, Education_Expenditure_GDP",
        "Mục_đích": "Kiểm tra quan hệ giữa tiếp cận giáo dục trung học và kết quả biết chữ.",
        "Nhận_xét_cần_kiểm_tra": "Các điểm lệch khỏi xu hướng chung là trường hợp đáng phân tích sâu.",
    },
    {
        "Worksheet": "EDU_04_Secondary_Heatmap",
        "Loại_biểu_đồ": "Heatmap",
        "Trường_chính": "Country_Name, Year, Secondary_Net_Enrollment",
        "Mục_đích": "Quan sát mô hình cải thiện, suy giảm hoặc thiếu hụt dữ liệu theo quốc gia và năm.",
        "Nhận_xét_cần_kiểm_tra": "Các mảng màu thấp/kéo dài giúp phát hiện quốc gia hoặc giai đoạn cần chú ý.",
    },
])
worksheet_plan


,Worksheet,Loại_biểu_đồ,Trường_chính,Mục_đích,Nhận_xét_cần_kiểm_tra
0,EDU_01_Enrollment_Trend,Line chart,"Year, Primary_Net_Enrollment, Secondary_Net_Enrollment",Theo dõi xu hướng nhập học tiểu học và trung học theo thời gian.,Tiểu học thường cao hơn trung học; khoảng cách giữa hai đường cho thấy rào cản chuyển tiếp.
1,EDU_02_Expenditure_By_Region,Bar chart,"Region, Education_Expenditure_GDP, Year",So sánh mức chi tiêu giáo dục giữa các vùng hoặc nhóm thu nhập.,Vùng chi tiêu cao chưa chắc có literacy/enrollment cao nếu dữ liệu có độ trễ hoặc thiếu.
2,EDU_03_Literacy_vs_Secondary,Scatter plot,"Secondary_Net_Enrollment, Adult_Literacy_Rate, Education_Expenditure_GDP",Kiểm tra quan hệ giữa tiếp cận giáo dục trung học và kết quả biết chữ.,Các điểm lệch khỏi xu hướng chung là trường hợp đáng phân tích sâu.
3,EDU_04_Secondary_Heatmap,Heatmap,"Country_Name, Year, Secondary_Net_Enrollment","Quan sát mô hình cải thiện, suy giảm hoặc thiếu hụt dữ liệu theo quốc gia và năm.",Các mảng màu thấp/kéo dài giúp phát hiện quốc gia hoặc giai đoạn cần chú ý.


## Kết luận cho công việc của Quốc Việt

- **2.1:** Đã khảo sát và xác nhận 4/4 indicator giáo dục trong WDI.
- **2.2:** Đã tiền xử lý dữ liệu giáo dục, loại aggregate rows, xử lý missing values thận trọng và xuất `data_output/wdi_education.csv`.
- **2.3:** Đã giải thích ý nghĩa từng metric và lý do chọn trong notebook.
- **2.4:** Đã đề xuất 4 worksheet Tableau phù hợp: line chart, bar chart, scatter plot và heatmap.
- **2.5:** Đã chuẩn bị các hướng nhận xét cần kiểm tra khi dựng biểu đồ.
- **2.7:** Đã tính thống kê mô tả cơ bản cho các indicator giáo dục.
- **2.9:** Đã chuẩn bị tài liệu bảng màu và formatting chung tại `tableau_specs/Nhom05_Color_Palette.md`.

Phần báo cáo chính không viết trong notebook này để nhóm trưởng tổng hợp và điền vào báo cáo chung sau.
